# 06 — Q-learning con capacidad operativa V2

Entrena Q-learning tabular y lo compara con las políticas Aleatoria, Reglas simples y Reglas escalonadas. La cohorte mantiene únicamente riesgo **bajo, medio y alto**; no se crea una cuarta categoría.

> Es un experimento de simulación metodológica, no una recomendación clínica.


In [ ]:
from pathlib import Path
import os, sys, subprocess
REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = 'feature/entorno-capacidad-v2'
PROJECT_ROOT = Path('/content/pida-rl-datos-publicos')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'gymnasium', 'pandas'])
if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)])
else:
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## Configuración

Se separan las semillas de entrenamiento y evaluación para reducir fuga experimental. Los hiperparámetros son explícitos y ajustables.


In [ ]:
CAPACIDAD_MENSUAL = 100
COSTOS_ACCION = {0: 0, 1: 1, 2: 3, 3: 5}
HORIZONTE_MESES = 12
EPISODIOS_ENTRENAMIENTO = 1500
SEMILLA_ENTRENAMIENTO = 10000
SEMILLAS_EVALUACION = (42, 123, 2026)
ALPHA = 0.10
GAMMA = 0.95
EPSILON_INICIAL = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.995


## Cohorte pública procesada

Carga el mismo archivo producido por el notebook 03; no genera pacientes nuevos.


In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/PIDA-RL-Diabetes')
COHORT_PATH = ROOT / 'data' / 'processed' / 'cohorte_diabetes.csv'
OUTPUT_DIR = ROOT / 'results' / 'capacidad_v2' / 'q_learning'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cohorte = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})
assert set(cohorte['categoria_riesgo'].unique()) <= {'bajo', 'medio', 'alto'}
print(cohorte.shape, cohorte['categoria_riesgo'].value_counts().to_dict())


## Entrenamiento

La tabla usa riesgo ordinal, `score_riesgo`, tiempo sin contacto, mes y recursos restantes. La capacidad forma parte del estado.


In [ ]:
from src.environment.diabetes_followup_env import DiabetesFollowUpEnv
from src.policies.q_learning import entrenar_q_learning

def crear_entorno():
    return DiabetesFollowUpEnv(cohorte, CAPACIDAD_MENSUAL, COSTOS_ACCION, HORIZONTE_MESES)

agente, historial = entrenar_q_learning(
    crear_entorno, EPISODIOS_ENTRENAMIENTO, SEMILLA_ENTRENAMIENTO,
    ALPHA, GAMMA, EPSILON_INICIAL, EPSILON_MIN, EPSILON_DECAY,
)
historial.to_csv(OUTPUT_DIR / 'historial_entrenamiento_q_learning.csv', index=False)
agente.tabla_dataframe().to_csv(OUTPUT_DIR / 'tabla_q.csv', index=False)
agente.guardar(OUTPUT_DIR / 'q_learning_v2.npz')
display(historial.tail())


## Evaluación común

Las cuatro políticas se evalúan en entornos nuevos y con las mismas semillas; Q-learning actúa sin exploración.


In [ ]:
from src.evaluation.evaluate_policies import evaluar_politicas
from src.evaluation.reporting import construir_reporte, exportar_reporte_csv
from src.policies.baselines import POLITICAS_BASE
from src.policies.q_learning import crear_politica_q_learning

politicas = dict(POLITICAS_BASE)
politicas['Q-learning'] = crear_politica_q_learning(agente)
resultados = evaluar_politicas(crear_entorno, politicas, SEMILLAS_EVALUACION)
reporte = construir_reporte(resultados)
exportar_reporte_csv(reporte, OUTPUT_DIR)
detalle = reporte['detalle']
assert set(detalle['categoria_riesgo']) <= {'bajo', 'medio', 'alto'}
assert (detalle['recursos_despues'] >= 0).all()
assert (detalle['recursos_usados_mes'] <= detalle['capacidad_mensual']).all()
display(reporte['resumen_algoritmos'].round(4))
display(reporte['clasificaciones_propuestas'].round(2))
display(reporte['clasificaciones_ejecutadas'].round(2))


## Interpretación responsable

- Compare recompensa, eventos, cobertura, recursos y ajustes por capacidad; no elija una política por una sola métrica.
- Las acciones propuestas pueden diferir de las ejecutadas cuando no hay recursos suficientes.
- Q-learning aprende dentro del simulador y debe reentrenarse si cambian estado, recompensa, costos o capacidad.
- Ningún resultado demuestra efectividad clínica real.
